# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [3]:
import sys; sys.path.append("/home/jovyan/tools")
from lakelib import *

## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [4]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 2,000,000 new records
write_files: 2,000,000 records -> /home/jovyan/data/lake/logs/run-20261123-140418-ebbc06

Lake check — topic 'logs' (4 partitions)   2026-11-23 14:06:03
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 517,691)     517,691           0          0           0
        1           [0, 495,136)     495,136           0          0           0
        2           [0, 467,910)     467,910           0          0           0
        3           [0, 519,263)     519,263           0          0           0
-------------------------------------------------------------------------------
    total      2,000,000 records   2,000,000           0          0           0

files under /home/jovyan/data/lake: 4 data files, 111.5 MB, 27.87 MB pe

{'topic': 2000000,
 'lake': 2000000,
 'dup': 0,
 'miss': 0,
 'lag': 0,
 'files': 4,
 'mb': 111.469501,
 'ok': True,
 'problems': []}

In [5]:
run_once()                        # the next run
check()

read_new: 2,000,000 new records
write_files: 2,000,000 records -> /home/jovyan/data/lake/logs/run-20261123-140603-5a47a7

Lake check — topic 'logs' (4 partitions)   2026-11-23 14:06:44
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 517,691)   1,035,382     517,691          0           0
        1           [0, 495,136)     990,272     495,136          0           0
        2           [0, 467,910)     935,820     467,910          0           0
        3           [0, 519,263)   1,038,526     519,263          0           0
-------------------------------------------------------------------------------
    total      2,000,000 records   4,000,000   2,000,000          0           0

files under /home/jovyan/data/lake: 8 data files, 222.9 MB, 27.87 MB per file on average
check took 1.4 s

Exa

{'topic': 2000000,
 'lake': 4000000,
 'dup': 2000000,
 'miss': 0,
 'lag': 0,
 'files': 8,
 'mb': 222.939002,
 'ok': False,
 'problems': ['2,000,000 duplicates']}

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Write `run_once()` for the architecture you chose (A, B or C, see the README).
The order of the calls *is* your architecture.

In [8]:
def run_once_A():
    # 1. where did the last run stop?
    start = load_state()              #Load start from state file
    # 2. read what is new
    new = read_new(start)           # Start reading from the last state 
    # 3. write it
    write_files(new, LAKE)          # Write new files directly to the LAKE
    # 4. remember where you stopped (if your architecture needs it)
    save_state(new.end)             # Saves where the run ended in state file

In [12]:
def run_once_B():
    # 1. where did the last run stop?
    start = lake_position()                    # Read the last position from the lake itself
    
    # 2. read what is new
    new = read_new(start)                      # Start reading from the last state 
    
    # 3. write it
    run = write_files(new, STAGING)            # Write new files to staging first 

    publish(run)                               # Move the completed run folder into lake
    # 4. remember where you stopped (if your architecture needs it)
    ...                                        # Architecture  B doesn't need to remember since it reads the start from the lake directly

In [20]:
def run_once_C():
    # 1. where did the last run stop?
    start = lake_position()                    # Read the last position from the lake itself
    
    # 2. read what is new
    new = read_new(start)                      # Start reading from the last state 
    
    # 3. write it
    write_files(new, LAKE)                     # Write directly into LAKE (duplicates from partial crashes are allowed)
    
    # 4. remember where you stopped (if your architecture needs it)
    ...                                        # # Architecture  C also reads the start from the lake directly



def run_once_C2():
    # 1. where did the last run stop?
    start = load_state()              #Load start from state file
    
    # 2. read what is new
    new = read_new(start)           # Start reading from the last state 
    
    # 3. write it
    write_files(new, LAKE)          # Write new files directly to the LAKE
    
    # 4. remember where you stopped (if your architecture needs it)
    save_state(new.end)             # Saves where the run ended in state file

In [16]:
def run_once():
    # 1. where did the last run stop?
    start = lake_position()                    # Read the last position from the lake itself
    
    # 2. read what is new
    new = read_new(start)                      # Start reading from the last state 
    
    # 3. write it
    run = write_files(new, STAGING)            # Write new files to staging first 

    publish(run)                               # Move the completed run folder into lake
    # 4. remember where you stopped (if your architecture needs it)
    ...                                        # Architecture  B doesn't need to remember since it reads the start from the lake directly

## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [9]:
check_all(run_once_A)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,000,000           0        0           0  35.7s  PASS
2 new records arrive, run again         2,100,000           0        0           0   9.5s  PASS
3 nothing new, run again                2,100,000           0        0           0   0.1s  PASS
4 run stops WHILE writing: right after  2,144,330           0        0      55,670         PASS
  ... then the next run                 2,244,330      44,330        0           0   7.4s  FAIL
5 run stops AFTER writing: right after  2,344,330      44,330        0           0         FAIL
  ... then the next run                 2,444,330     144,330        0           0   7.6s  FAIL
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 2000000,
   'lake': 2000000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 111.469501,
   'ok': True,
   'problems': []},
  35.66197371482849),
 ('2 new records arrive, run again',
  {'topic': 2100000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 112.627979,
   'ok': True,
   'problems': []},
  9.479491949081421),
 ('3 nothing new, run again',
  {'topic': 2100000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 112.627979,
   'ok': True,
   'problems': []},
  0.07693600654602051),
 ('4 run stops WHILE writing: right after',
  {'topic': 2200000,
   'lake': 2144330,
   'dup': 0,
   'miss': 0,
   'lag': 55670,
   'files': 9,
   'mb': 113.098657,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 2200000,
   'lake': 2244330,
   'dup': 44330,
   'miss': 0,
   'lag': 0,
   'files': 13,
   'mb': 114.251753,
   'ok': False,
   'problems': [

In [13]:
check_all(run_once_B)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,300,000           0        0           0  59.9s  PASS
2 new records arrive, run again         2,400,000           0        0           0   9.1s  PASS
3 nothing new, run again                2,400,000           0        0           0   0.7s  PASS
4 run stops WHILE writing: right after  2,400,000           0        0     100,000         PASS
  ... then the next run                 2,500,000           0        0           0   9.9s  PASS
5 run stops AFTER writing: right after  2,500,000           0        0     100,000         PASS
  ... then the next run                 2,600,000           0        0           0  10.5s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 2300000,
   'lake': 2300000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 116.653011,
   'ok': True,
   'problems': []},
  59.91027760505676),
 ('2 new records arrive, run again',
  {'topic': 2400000,
   'lake': 2400000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 117.811056,
   'ok': True,
   'problems': []},
  9.106446504592896),
 ('3 nothing new, run again',
  {'topic': 2400000,
   'lake': 2400000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 117.811056,
   'ok': True,
   'problems': []},
  0.7472035884857178),
 ('4 run stops WHILE writing: right after',
  {'topic': 2500000,
   'lake': 2400000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 8,
   'mb': 117.811056,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 2500000,
   'lake': 2500000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 118.968338,
   'ok': True,
   'problems': []},
 

In [15]:
check_all(run_once_C, lake_sql=DEDUP_SQL)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,600,000           0        0           0  60.4s  PASS
2 new records arrive, run again         2,700,000           0        0           0  10.0s  PASS
3 nothing new, run again                2,700,000           0        0           0   0.6s  PASS
4 run stops WHILE writing: right after  2,731,257           0        0      68,743         PASS
  ... then the next run                 2,800,000           0        0           0   8.0s  PASS
5 run stops AFTER writing: right after  2,900,000           0        0           0         PASS
  ... then the next run                 2,900,000           0        0           0  -0.5s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 2600000,
   'lake': 2600000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 121.819277,
   'ok': True,
   'problems': []},
  60.431546449661255),
 ('2 new records arrive, run again',
  {'topic': 2700000,
   'lake': 2700000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 122.975602,
   'ok': True,
   'problems': []},
  10.008160829544067),
 ('3 nothing new, run again',
  {'topic': 2700000,
   'lake': 2700000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 122.975602,
   'ok': True,
   'problems': []},
  0.5920588970184326),
 ('4 run stops WHILE writing: right after',
  {'topic': 2800000,
   'lake': 2731257,
   'dup': 0,
   'miss': 0,
   'lag': 68743,
   'files': 9,
   'mb': 123.324067,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 2800000,
   'lake': 2800000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 124.128131,
   'ok': True,
   'problems': []},


In [21]:
check_all(run_once_C2, lake_sql=DEDUP_SQL)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             3,500,000           0        0           0  65.4s  PASS
2 new records arrive, run again         3,600,000           0        0           0  10.3s  PASS
3 nothing new, run again                3,600,000           0        0           0   0.1s  PASS
4 run stops WHILE writing: right after  3,627,251           0        0      72,749         PASS
  ... then the next run                 3,700,000           0        0           0  10.2s  PASS
5 run stops AFTER writing: right after  3,800,000           0        0           0         PASS
  ... then the next run                 3,800,000           0        0           0  10.0s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 3500000,
   'lake': 3500000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 137.317583,
   'ok': True,
   'problems': []},
  65.36862683296204),
 ('2 new records arrive, run again',
  {'topic': 3600000,
   'lake': 3600000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 7,
   'mb': 138.420811,
   'ok': True,
   'problems': []},
  10.253618478775024),
 ('3 nothing new, run again',
  {'topic': 3600000,
   'lake': 3600000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 7,
   'mb': 138.420811,
   'ok': True,
   'problems': []},
  0.12216472625732422),
 ('4 run stops WHILE writing: right after',
  {'topic': 3700000,
   'lake': 3627251,
   'dup': 0,
   'miss': 0,
   'lag': 72749,
   'files': 8,
   'mb': 138.731844,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 3700000,
   'lake': 3700000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 139.885602,
   'ok': True,
   'problems': []},


In [17]:
check_all(run_once)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,900,000           0        0           0  59.1s  PASS
2 new records arrive, run again         3,000,000           0        0           0   9.4s  PASS
3 nothing new, run again                3,000,000           0        0           0   0.7s  PASS
4 run stops WHILE writing: right after  3,000,000           0        0     100,000         PASS
  ... then the next run                 3,100,000           0        0           0  10.6s  PASS
5 run stops AFTER writing: right after  3,100,000           0        0     100,000         PASS
  ... then the next run                 3,200,000           0        0           0   9.9s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 2900000,
   'lake': 2900000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 126.985555,
   'ok': True,
   'problems': []},
  59.06041979789734),
 ('2 new records arrive, run again',
  {'topic': 3000000,
   'lake': 3000000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 128.139653,
   'ok': True,
   'problems': []},
  9.43700909614563),
 ('3 nothing new, run again',
  {'topic': 3000000,
   'lake': 3000000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 128.139653,
   'ok': True,
   'problems': []},
  0.7296299934387207),
 ('4 run stops WHILE writing: right after',
  {'topic': 3100000,
   'lake': 3000000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 8,
   'mb': 128.139653,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 3100000,
   'lake': 3100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 129.296764,
   'ok': True,
   'problems': []},
  